# Inference

This continues from [the deployment notebook](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/bedrock/foundation_sec_8b_reasoning/deploy.ipynb). <br>
Let's perform inference using the reasoning model deployed on Amazon Bedrock.

Bedrock's `invoke_model` API accepts OpenAI-style messages for imported models, so no tokenizer or chat template formatting is needed. Since this is a reasoning model, the response contains a `<think>...</think>` reasoning trace followed by the final answer.

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3
import json
import re

In [ ]:
AWS_REGION = "us-west-2"
MODEL_ARN = ""  # copy from deploy notebook or: aws bedrock list-imported-models
assert MODEL_ARN, "Set MODEL_ARN above before running inference"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
max_tokens = 8192

In [ ]:
def parse_reasoning(raw_output: str) -> dict:
    """Split model output at </think> into reasoning trace and final answer."""
    match = re.search(r"</think>", raw_output, re.IGNORECASE)
    if match:
        reasoning = raw_output[:match.start()]
        content = raw_output[match.end():]
        reasoning = re.sub(r"^\s*<think>\s*", "", reasoning, flags=re.IGNORECASE)
        return {
            "reasoning_content": reasoning.strip(),
            "content": content.strip(),
        }
    return {
        "reasoning_content": None,
        "content": raw_output.strip(),
    }

In [ ]:
def chat_generation(user_prompt):
    body = {
        "messages": [
            {"role": "user", "content": user_prompt}
        ],
        "max_tokens": max_tokens,
        "temperature": 0.6,
        "top_p": 0.95,
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ARN,
        body=json.dumps(body),
    )
    response_body = json.loads(response["body"].read())

    raw_generation = response_body["choices"][0]["message"]["content"]
    parsed = parse_reasoning(raw_generation)

    return parsed["reasoning_content"], parsed["content"]

In [ ]:
user_prompt = "What is cybersecurity?"

reasoning, answer = chat_generation(user_prompt)

print("=== Reasoning (chain-of-thought) ===")
print(reasoning)
print()
print("=== Answer ===")
print(answer)